# Lab | Bias Clinic and Sampling Variability

## Task 1: Bias Diagnosis — Case Studies

### Scenario A — App Review Analysis
**Type of Bias:** Self-Selection Bias / Response Bias.  
**How it distorts conclusions:** Users who leave reviews are often those with extreme experiences (either very positive or very negative). Passive, moderately satisfied, or slightly dissatisfied users may not leave reviews at all. Relying solely on these reviews overestimates general user satisfaction because the sample is not representative of the entire user base.  
**Concrete Fix:** Implement an in-app prompt that randomly surveys a representative cross-section of all active users, including those who wouldn't normally seek out the review page.

### Scenario B — Startup Success Study
**Type of Bias:** Survivorship Bias.  
**How it distorts conclusions:** By only looking at "successful" and "still operating" startups, the study ignores the thousands of startups that pivoted and still failed, or those that didn't pivot and failed. It creates a false correlation between pivoting and success because it doesn't account for the "graveyard" of failed attempts.  
**Concrete Fix:** Include data from both successful and failed startups founded within the same timeframe to determine if pivoting actually differentiates survivors from those that went under.

### Scenario C — Health Survey
**Type of Bias:** Non-Response Bias / Volunteer Bias.  
**How it distorts conclusions:** With only a 10% response rate, the respondents likely differ significantly from the non-respondents. People who are health-conscious are more likely to take the time to answer a health survey, leading to an overestimation of average exercise hours and self-reported health in the general subscriber population.  
**Concrete Fix:** Follow up with a random sample of the non-respondents using incentives (like a small gift card) to ensure their habits are captured, or use weighted adjustments based on known demographics of the full subscriber list.

### Scenario D — Salary Benchmarking
**Type of Bias:** Selection Bias / Sampling Bias.  
**How it distorts conclusions:** The data is skewed toward tech workers in large, high-cost-of-living cities where salaries are naturally higher. A company in a small town using this data will likely overpay significantly relative to their local market, as the benchmark doesn't reflect the geographic or industry diversity relevant to their specific location.  
**Concrete Fix:** Filter the benchmarking data by geographic region or use a cost-of-living adjustment factor to normalize the salary data for the local market.

## Task 2: Create the Population

In this task, we generate a synthetic population of 100,000 individuals to serve as our "ground truth" for the sampling experiments.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
sns.set_style("whitegrid")
1
n_population = 100000

age = np.random.normal(loc=40, scale=15, size=n_population)
age = np.clip(age, 18, 85).astype(int)

income = 1500 * age + np.random.normal(0, 20000, size=n_population)
income = np.clip(income, 15000, 250000)

satisfaction = (income / 25000) + np.random.normal(0, 2, size=n_population)
satisfaction = np.clip(satisfaction, 1, 10)

regions = ["Urban", "Suburban", "Rural"]
region = np.random.choice(regions, size=n_population, p=[0.60, 0.25, 0.15])

population_df = pd.DataFrame({
    'age': age,
    'income': income,
    'satisfaction': satisfaction,
    'region': region
})

true_parameters = {
    'mean_age': population_df['age'].mean(),
    'mean_income': population_df['income'].mean(),
    'mean_satisfaction': population_df['satisfaction'].mean(),
    'prop_urban': (population_df['region'] == 'Urban').mean(),
    'prop_suburban': (population_df['region'] == 'Suburban').mean(),
    'prop_rural': (population_df['region'] == 'Rural').mean()
}

print("True Population Parameters:")
for param, value in true_parameters.items():
    print(f"{param}: {value:.2f}")

print("\nPopulation Summary:")
print(population_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(population_df['age'], bins=30, ax=axes[0], kde=True).set_title('Age Distribution')
sns.histplot(population_df['income'], bins=30, ax=axes[1], kde=True).set_title('Income Distribution')
sns.histplot(population_df['satisfaction'], bins=10, ax=axes[2], kde=True).set_title('Satisfaction Distribution')
plt.tight_layout()
plt.show()

## Task 3: Biased vs. Unbiased Sampling

Now we draw samples from our population using different strategies and compare them to the ground truth.

In [ ]:
def get_sample_stats(sample_df):
    return {
        'mean_age': sample_df['age'].mean(),
        'mean_income': sample_df['income'].mean(),
        'mean_satisfaction': sample_df['satisfaction'].mean(),
        'prop_urban': (sample_df['region'] == 'Urban').mean(),
        'prop_suburban': (sample_df['region'] == 'Suburban').mean(),
        'prop_rural': (sample_df['region'] == 'Rural').mean()
    }

n_sample = 200

srs_sample = population_df.sample(n_sample, random_state=42)

urban_sample = population_df[population_df['region'] == 'Urban'].sample(n_sample, random_state=42)

income_median = population_df['income'].median()
high_income_sample = population_df[population_df['income'] > income_median].sample(n_sample, random_state=42)

results = {
    'True Population': true_parameters,
    'SRS Sample': get_sample_stats(srs_sample),
    'Urban Only': get_sample_stats(urban_sample),
    'High Income': get_sample_stats(high_income_sample)
}

comparison_df = pd.DataFrame(results).T
print("Comparison of Sample Statistics:")
display(comparison_df)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.kdeplot(population_df['income'], label='Population', color='black', linestyle='--')
sns.kdeplot(srs_sample['income'], label='SRS', color='blue')
sns.kdeplot(urban_sample['income'], label='Urban Only', color='green')
sns.kdeplot(high_income_sample['income'], label='High Income', color='red')
plt.title('Income KDE Comparison')
plt.legend()

plt.subplot(1, 2, 2)
sns.kdeplot(population_df['satisfaction'], label='Population', color='black', linestyle='--')
sns.kdeplot(srs_sample['satisfaction'], label='SRS', color='blue')
sns.kdeplot(urban_sample['satisfaction'], label='Urban Only', color='green')
sns.kdeplot(high_income_sample['satisfaction'], label='High Income', color='red')
plt.title('Satisfaction KDE Comparison')
plt.legend()

plt.tight_layout()
plt.show()

### Repeated Sampling (1,000 times)

We now simulate drawing 1,000 samples for each strategy to see the sampling distribution of the mean income.

In [ ]:
n_iterations = 1000
srs_means = []
urban_means = []
high_income_means = []

for _ in range(n_iterations):
    srs_means.append(population_df['income'].sample(n_sample).mean())
    urban_means.append(population_df[population_df['region'] == 'Urban']['income'].sample(n_sample).mean())
    high_income_means.append(population_df[population_df['income'] > income_median]['income'].sample(n_sample).mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

sns.histplot(srs_means, ax=axes[0], color='blue', kde=True)
axes[0].axvline(true_parameters['mean_income'], color='black', linestyle='--', label='True Mean')
axes[0].set_title('SRS Sampling Distribution')
axes[0].legend()

sns.histplot(urban_means, ax=axes[1], color='green', kde=True)
axes[1].axvline(true_parameters['mean_income'], color='black', linestyle='--', label='True Mean')
axes[1].set_title('Urban Only Sampling Distribution')
axes[1].legend()

sns.histplot(high_income_means, ax=axes[2], color='red', kde=True)
axes[2].axvline(true_parameters['mean_income'], color='black', linestyle='--', label='True Mean')
axes[2].set_title('High Income Sampling Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

**Guiding question:** Which sampling strategies produce biased estimates? How can you tell from the sampling distributions?

- **Simple Random Sampling (SRS):** This strategy is **unbiased**. The sampling distribution is centered directly around the true population mean. Any individual sample might deviate due to sampling variability, but on average, it hits the target.
- **Urban Only:** This strategy shows **little to no bias for income** in this specific simulation because region was assigned randomly and not correlated with income. However, it is **biased for regional proportions** (it thinks the population is 100% urban). In reality, if urban areas had higher costs or salaries, this would be heavily biased for income too.
- **High-Income Filter:** This strategy is **heavily biased**. The sampling distribution is centered far to the right of the true population mean. This is because we systematically excluded the bottom 50% of the income distribution, leading to a massive overestimation of the average income.

## Task 4: Bootstrapping

Bootstrapping allows us to estimate the uncertainty of a statistic (like the mean) by resampling from our observed data *with replacement*.

In [ ]:
observed_sample = population_df.sample(200, random_state=42)
observed_mean = observed_sample['income'].mean()

n_bootstrap = 1000
bootstrap_means = []
for _ in range(n_bootstrap):
    bootstrap_sample = observed_sample['income'].sample(len(observed_sample), replace=True)
    bootstrap_means.append(bootstrap_sample.mean())

ci_lower = np.percentile(bootstrap_means, 2.5)
ci_upper = np.percentile(bootstrap_means, 97.5)

print(f"Observed Sample Mean: {observed_mean:.2f}")
print(f"95% Bootstrap Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"True Population Mean: {true_parameters['mean_income']:.2f}")

plt.figure(figsize=(10, 6))
sns.histplot(bootstrap_means, kde=True, color='purple')
plt.axvline(observed_mean, color='blue', linestyle='-', label='Observed Mean')
plt.axvline(ci_lower, color='red', linestyle='--', label='95% CI')
plt.axvline(ci_upper, color='red', linestyle='--')
plt.axvline(true_parameters['mean_income'], color='black', linestyle=':', label='True Population Mean')
plt.title('Bootstrap Distribution of the Mean Income')
plt.legend()
plt.show()

## Task 5: Data Leakage Demonstration

Data leakage occurs when information from outside the training dataset is used to create the model. A classic example is scaling the data before splitting.

In [ ]:
df_leakage = population_df.copy()
median_sat = df_leakage['satisfaction'].median()
df_leakage['high_satisfaction'] = (df_leakage['satisfaction'] > median_sat).astype(int)
X = df_leakage[['age', 'income']]
y = df_leakage['high_satisfaction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
acc_proper = accuracy_score(y_test, y_pred)

scaler_leaked = StandardScaler()
X_scaled_leaked = scaler_leaked.fit_transform(X)
X_train_L, X_test_L, y_train_L, y_test_L = train_test_split(X_scaled_leaked, y, test_size=0.2, random_state=42)

model_leaked = LogisticRegression()
model_leaked.fit(X_train_L, y_train_L)
y_pred_L = model_leaked.predict(X_test_L)
acc_leaked = accuracy_score(y_test_L, y_pred_L)

print(f"Accuracy (Proper): {acc_proper:.5f}")
print(f"Accuracy (With Leakage): {acc_leaked:.5f}")
print(f"Difference: {acc_leaked - acc_proper:.5f}")

print("\nNote: In small datasets or specific distributions, leakage can significantly inflate performance metrics,")
print("leading to models that perform poorly in production despite great test scores.")